# Resumen Sachs: Beta=0 vs Beta óptima (kan y kaam)

Analiza, para los modelos `kan` y `kaam` sobre el caso semi-sintético **Sachs** (`notebooks/Experimento5/sachs_hsic_beta.ipynb`),
si la mejora del modelo con término HSIC (`kan_hsic`/`kaam_hsic`, `loss='hybrid'` con la `Beta` óptima) frente
al modelo base (`Beta=0`, `loss='mse'`) es significativa, con un test de Wilcoxon pareado -- igual que en
`Resumen_Observacional.ipynb` / `Resumen_Intervencional.ipynb` / `Resumen_Sinteticos_*.ipynb`.

**Requisito de orden**: este notebook lee los pickles que genera `sachs_hsic_beta.ipynb`
(`outputs/sachs_hsic/data/sachs_{beta_sweep,results}_*.pkl`), y recalcula aquí la misma Beta óptima que
esa selecciona (mismo criterio multi-métrica) solo para dejarla explícita en la tabla -- si se ejecuta
este notebook con pickles de una ejecución anterior de `sachs_hsic_beta.ipynb` que usara otro criterio
de selección, la Beta mostrada aquí puede no coincidir con la que realmente se entrenó en
`kan_hsic`/`kaam_hsic`. Ejecutar siempre `sachs_hsic_beta.ipynb` primero.

**Diferencia clave frente a los demás `Resumen_*.ipynb`**: Sachs no se evaluó con múltiples semillas
independientes (los datos factuales se generaron una única vez con `seed=42`, ver `sachs_hsic_beta.ipynb`),
así que aquí no existe una columna `Seed` por la que emparejar. En su lugar:

- El barrido de `Beta` (`sachs_beta_sweep_{kan,kaam}.pkl`) solo registra métricas observacionales agregadas
  (`mmd_obs`, `rf_acc_obs`) por valor de `Beta`, sin repeticiones. La Beta óptima se selecciona con el
  mismo criterio multi-métrica que `Resumen_Observacional.ipynb` / `Resumen_Sinteticos_Observacional.ipynb`
  (función `beta_optima_criterio`, celda siguiente), aplicado sobre `MMD Obs` + `RF Acc Obs`.
- Los modelos finales (`sachs_results_{kan,kaam,kan_hsic,kaam_hsic}.pkl`) sí guardan, para las métricas
  interventional/counterfactual, un valor por cada una de las **660 intervenciones evaluadas** (11 nodos x 60
  valores de `do(·)` en `[-3, 3)`, ver `inter_vector` en `sachs_hsic_beta.ipynb`) -- y esas 660 intervenciones
  son las mismas para todos los modelos. Eso da un emparejamiento natural (por intervención, no por semilla)
  para `MMD Int`, `RF Acc Int`, `MSE CF` y `MAE CF`, sobre el que sí se puede aplicar Wilcoxon pareado.
- Las métricas puramente observacionales (`MMD Obs`, `RF Acc Obs`) son un único valor agregado por modelo
  (no hay repeticiones que emparejar), así que se muestran como referencia pero **sin** test de significancia.

In [1]:
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import wilcoxon

pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 20)

REPO_ROOT = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
NOTEBOOKS_DIR = REPO_ROOT / "notebooks"
DATA_DIR = REPO_ROOT / "outputs" / "sachs_hsic" / "data"

ALPHA = 0.05
FAMILIAS = ["kan", "kaam"]
BETAS = [round(0.1 * i, 1) for i in range(1, 9)]  # 0.1 .. 0.8, igual que en sachs_hsic_beta.ipynb

## Carga de resultados (`outputs/sachs_hsic/data`)

Para cada familia se cargan tres pickles generados por `sachs_hsic_beta.ipynb`:
- `sachs_beta_sweep_<familia>.pkl`: barrido de `Beta` en `[0.1, 0.8]` (`mmd_obs`, `rf_acc_obs` por beta).
- `sachs_results_<familia>.pkl`: modelo base (`Beta=0`, `loss='mse'`).
- `sachs_results_<familia>_hsic.pkl`: modelo con la Beta óptima (`loss='hybrid'`).

In [2]:
def cargar_pickle(nombre: str):
    path = DATA_DIR / f"{nombre}.pkl"
    assert path.exists(), f"No existe: {path}"
    with open(path, "rb") as f:
        return pickle.load(f)


beta_sweep = {familia: cargar_pickle(f"sachs_beta_sweep_{familia}") for familia in FAMILIAS}

resultados_beta0 = {familia: cargar_pickle(f"sachs_results_{familia}") for familia in FAMILIAS}
resultados_hsic = {familia: cargar_pickle(f"sachs_results_{familia}_hsic") for familia in FAMILIAS}

## Selección de la Beta óptima

Mismo criterio multi-métrica que `Resumen_Observacional.ipynb` / `Resumen_Sinteticos_Observacional.ipynb`
(no el simple mínimo de una sola métrica): para cada métrica (`MMD Obs`, `RF Acc Obs`) se halla la Beta > 0
que la minimiza en el barrido `[0.1, 0.8]` (candidatas), y se desempata por (a) menos métricas empeoradas
frente a `Beta=0`, (b) más métricas mejoradas, (c) mayor mejora relativa total, (d) mayor mejora relativa
en una única métrica. `Beta=0` (`sachs_results_<familia>.pkl`) es solo la referencia/baseline, nunca
candidata. Esto es exactamente lo que hace `sachs_hsic_beta.ipynb` (función `beta_optima_criterio`) para
decidir con qué Beta entrenar `kan_hsic`/`kaam_hsic`; se recalcula aquí para mostrarla explícitamente.

In [3]:
def beta_optima_criterio(valores_por_beta: pd.DataFrame, metrics=("mmd_obs", "rf_acc_obs")):
    '''Selecciona la Beta optima (Beta != 0) con el mismo criterio multi-metrica que
    Resumen_Observacional.ipynb / Resumen_Sinteticos_Observacional.ipynb / sachs_hsic_beta.ipynb: para
    cada metrica se halla la Beta > 0 que la minimiza (candidatas); se desempata por (a) menos metricas
    empeoradas frente a Beta=0, (b) mas metricas mejoradas, (c) mayor mejora relativa total, (d) mayor
    mejora relativa en una unica metrica.

    `valores_por_beta` debe tener indice = Beta (incluyendo 0.0 como baseline) y columnas = metrics.
    Devuelve (beta_opt: float, candidatas: pd.DataFrame).
    '''
    baseline = valores_por_beta.loc[0.0]
    no_cero = valores_por_beta.drop(index=0.0)

    betas_candidatas = sorted(set(no_cero[m].idxmin() for m in metrics))

    filas = []
    for beta in betas_candidatas:
        valores = no_cero.loc[beta]
        mejora_relativa = (baseline - valores) / baseline
        filas.append({
            "Beta": beta,
            "n_mejoradas": int((valores < baseline).sum()),
            "n_empeoradas": int((valores > baseline).sum()),
            "mejora_relativa_total": float(mejora_relativa.sum()),
            "mejora_relativa_max": float(mejora_relativa.max()),
        })

    candidatas = pd.DataFrame(filas).sort_values(
        by=["n_empeoradas", "n_mejoradas", "mejora_relativa_total", "mejora_relativa_max"],
        ascending=[True, False, False, False],
    ).reset_index(drop=True)

    return float(candidatas.iloc[0]["Beta"]), candidatas


beta_optima = {}
candidatas_beta_optima = {}
for familia in FAMILIAS:
    valores_por_beta = pd.DataFrame({
        "mmd_obs": {0.0: resultados_beta0[familia]["mmd_obs_avg"], **{b: beta_sweep[familia][b]["mmd_obs"] for b in BETAS}},
        "rf_acc_obs": {0.0: resultados_beta0[familia]["rf_acc_obs_avg"], **{b: beta_sweep[familia][b]["rf_acc_obs"] for b in BETAS}},
    })
    beta_optima[familia], candidatas_beta_optima[familia] = beta_optima_criterio(valores_por_beta)
    print(f"Beta optima ({familia}): {beta_optima[familia]}")

Beta optima (kan): 0.3
Beta optima (kaam): 0.3


## Métricas y emparejamiento

- `METRICAS_PAREADAS`: métricas interventional/counterfactual, con un valor por cada una de las 660
  intervenciones (`<clave>_all` en los pickles) -- estas sí se comparan con Wilcoxon pareado.
- `METRICAS_PUNTUALES`: métricas observacionales agregadas (`<clave>_avg`), un único valor por modelo --
  se muestran para contexto pero sin p-valor.

En las 6 métricas, menor es mejor.

In [4]:
METRICAS_PAREADAS = {
    "MMD Int": "mmd_int_all",
    "RF Acc Int": "rf_acc_int_all",
    "MSE CF": "mse_cf_all",
    "MAE CF": "mae_cf_all",
}
METRICAS_PUNTUALES = {
    "MMD Obs": "mmd_obs_avg",
    "RF Acc Obs": "rf_acc_obs_avg",
}

## Función de test de Wilcoxon pareado (por intervención)

Para cada métrica pareada, se comparan los 660 valores del modelo base (`Beta=0`) frente a los 660 del
modelo con la Beta óptima, emparejados por posición (misma intervención `(nodo, valor)` en ambos, ya que
`inter_vector` es común a todos los modelos en `sachs_hsic_beta.ipynb`). Test unidireccional
(`alternative='less'`): H0 = no hay diferencia; H1 = el valor con la Beta óptima es menor (mejor) que con
Beta=0.

In [5]:
def wilcoxon_sachs(base: dict, hsic: dict, metrics=METRICAS_PAREADAS):
    '''Wilcoxon signed-rank pareado (por intervención) entre Beta=0 y Beta=beta_optima, por métrica.

    H0: no hay diferencia. H1 (alternative='less'): el valor en beta_optima es menor (mejor) que en Beta=0.
    Devuelve (p_valores: dict metrica->p, n_pares: int).
    '''
    p_valores = {}
    n_pares = None
    for nombre, clave in metrics.items():
        base_vals = np.asarray(base[clave], dtype=float)
        hsic_vals = np.asarray(hsic[clave], dtype=float)
        assert len(base_vals) == len(hsic_vals), f"Longitudes distintas para {nombre}"
        n_pares = len(base_vals)
        try:
            _, p = wilcoxon(hsic_vals, base_vals, alternative="less")
        except ValueError:
            p = float("nan")
        p_valores[nombre] = p
    return p_valores, n_pares

## Cálculo (tabla resumen + p-valores) para kan y kaam

In [6]:
filas = []
for familia in FAMILIAS:
    base = resultados_beta0[familia]
    hsic = resultados_hsic[familia]
    p_valores, n_pares = wilcoxon_sachs(base, hsic)

    fila = {"Familia": familia.upper(), "beta_optima": beta_optima[familia]}
    for nombre, clave in METRICAS_PUNTUALES.items():
        fila[f"{nombre} beta=0"] = base[clave]
        fila[f"{nombre} beta_optima"] = hsic[clave]
    for nombre, clave in METRICAS_PAREADAS.items():
        fila[f"{nombre} beta=0"] = float(np.mean(base[clave]))
        fila[f"{nombre} beta_optima"] = float(np.mean(hsic[clave]))
        fila[f"{nombre} p_valor"] = p_valores[nombre]
    fila["n_pares"] = n_pares
    filas.append(fila)

resumen = pd.DataFrame(filas).set_index("Familia")

## Funciones de formato y resaltado en negrita

In [7]:
COLUMNA_A_METRICA = {f"{nombre} beta_optima": nombre for nombre in METRICAS_PAREADAS}


def formatear(resumen: pd.DataFrame):
    cols_num = [c for c in resumen.columns if c not in ("beta_optima",)]
    fmt = resumen.copy()
    fmt[cols_num] = fmt[cols_num].round(5)
    fmt["beta_optima"] = fmt["beta_optima"].round(2)
    return fmt


def tabla_con_negrita(resumen: pd.DataFrame):
    '''Tabla resumen con las celdas "<metrica> beta_optima" en negrita cuando su p-valor (Wilcoxon) < ALPHA.

    Solo aplica en la visualización del notebook (un CSV plano no admite negrita). Las columnas de
    METRICAS_PUNTUALES nunca se resaltan (no tienen p-valor: no hay repeticiones que emparejar).
    '''
    fmt = formatear(resumen)

    def resaltar(row):
        estilos = []
        for col in row.index:
            metrica = COLUMNA_A_METRICA.get(col)
            if metrica is not None and pd.notna(row[f"{metrica} p_valor"]) and row[f"{metrica} p_valor"] < ALPHA:
                estilos.append("font-weight: bold")
            else:
                estilos.append("")
        return estilos

    return fmt.style.apply(resaltar, axis=1)

## Resultados: kan y kaam, Beta=0 vs Beta óptima

En negrita, las celdas `beta_optima` cuya mejora frente a `Beta=0` es significativa (Wilcoxon pareado
por intervención, p < 0.05). Solo aplica a `MMD Int`, `RF Acc Int`, `MSE CF` y `MAE CF`: `MMD Obs`/`RF Acc Obs`
se muestran sin test (un único valor agregado, sin repeticiones que emparejar).

In [8]:
tabla_con_negrita(resumen)

,beta_optima,MMD Obs beta=0,MMD Obs beta_optima,RF Acc Obs beta=0,RF Acc Obs beta_optima,MMD Int beta=0,MMD Int beta_optima,MMD Int p_valor,RF Acc Int beta=0,RF Acc Int beta_optima,RF Acc Int p_valor,MSE CF beta=0,MSE CF beta_optima,MSE CF p_valor,MAE CF beta=0,MAE CF beta_optima,MAE CF p_valor,n_pares
Familia,,,,,,,,,,,,,,,,,,
KAN,0.300000,0.007960,0.007790,0.575000,0.550000,0.019710,0.019650,0.000000,0.565590,0.577420,1.000000,0.114480,0.117160,0.678100,0.081470,0.082250,0.497310,660
KAAM,0.300000,0.007960,0.008070,0.591670,0.591670,0.019960,0.020720,1.000000,0.565760,0.581680,1.000000,0.116350,0.124670,1.000000,0.083440,0.086220,1.000000,660


In [9]:
cols_p = [c for c in resumen.columns if c.endswith("p_valor")] + ["n_pares"]
resumen[cols_p].round(4)

,MMD Int p_valor,RF Acc Int p_valor,MSE CF p_valor,MAE CF p_valor,n_pares
Familia,,,,,
KAN,0.0,1.0,0.6781,0.4973,660
KAAM,1.0,1.0,1.0000,1.0000,660


## Guardar CSV resumen

In [10]:
OUT_PATH = NOTEBOOKS_DIR / "tablas" / "resumen_sachs_beta.csv"
resumen.reset_index().to_csv(OUT_PATH, index=False)
print(f"Guardado en: {OUT_PATH}")

Guardado en: c:\Users\aarna\Desktop\clau\TFM Claudia\kacgm-hsic\notebooks\tablas\resumen_sachs_beta.csv
